In [3]:
%pip install spacy


  Using cached spacy-3.8.11-cp312-cp312-win_amd64.whl.metadata (28 kB)
Using cached spacy-3.8.11-cp312-cp312-win_amd64.whl (14.2 MB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\waqas\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\spacy\\lang\\da\\stop_words.py'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
   ---------------------------------------- 0.0/14.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.2 MB ? eta -:--:--
    --------------------------------------- 0.3/14.2 MB ? eta -:--:--
    --------------------------------------- 0.3/14.2 MB ? eta -:--:--
   -- ------------------------------------- 0.8/14.2 MB 1.2 MB/s eta 0:00:12
   -- ------------------------------------- 0.8/14.2 MB 1.2 MB/s eta 0:00:12
   -- ------------------------------------- 1.0/14.2 MB 883.6 kB/s eta 0:00:15
   -- ------------------------------------- 1.0/14.2 MB 883.6 kB/s eta 0:00:15
   -- ------------------------------------- 1.0/14.2 MB 883.6 kB/s eta 0:00:15
   --- ------------------------------------ 1.3/14.2 MB 780.8 kB/s eta 0:00:17
   ---- ----------------------------------- 1.6/14.2 MB 791.5 kB/s eta 0:00:16
   ---- ---------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     - ------------------------------------- 0.5/12.8 MB 837.5 kB/s eta 0:00:15
     -- ------------------------------------- 0.8/12.8 MB 1.1 MB/s eta 0:00:12
     --- ------------------------------------ 1.0/12.8 MB 1.0 MB/s eta 0:00:12
     --- ------------------------------------ 1.0/12.8 MB 1.0 MB/s eta 0:00:12
     --- ------------------------------------ 1.0/12.8 MB 1.0 MB/s eta 0:00:12
     --- ----------------------------------- 1.3/12.8 MB 771.0 kB/s eta 0:00:15
     ----- --------------------------------- 1.8/12.8 MB 940.7 kB/s eta 0:00:12
     ----- --------------------------------- 1.8/12.8 MB 940.7 kB/s eta 0:00:12
     ------- ------------------------------- 2.4/12.8 MB 986.9 kB/s eta 0:00


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import spacy
import pandas as pd
from spacy import displacy
from collections import Counter
import os

In [11]:
# Path to your dataset folder
folder_path = r"D:\Internship\Task_3"

# Function to load CoNLL-style NER files
def load_ner_file(file_path):
    sentences = []
    labels = []

    with open(file_path, "r", encoding="utf-8") as f:
        sentence = []
        label = []
        for line in f:
            line = line.strip()
            if not line:  # Empty line = end of sentence
                if sentence:
                    sentences.append(sentence)
                    labels.append(label)
                    sentence = []
                    label = []
            else:
                parts = line.split()      # Usually: word [other columns] label
                word = parts[0]           # first column = word
                ner_label = parts[-1]     # last column = NER label
                sentence.append(word)
                label.append(ner_label)
        # Add last sentence if file doesn't end with newline
        if sentence:
            sentences.append(sentence)
            labels.append(label)

    return sentences, labels

# Load train, valid, and test files
train_sentences, train_labels = load_ner_file(os.path.join(folder_path, "train.txt"))
valid_sentences, valid_labels = load_ner_file(os.path.join(folder_path, "valid.txt"))
test_sentences, test_labels = load_ner_file(os.path.join(folder_path, "test.txt"))

# Quick check
print("Number of training sentences:", len(train_sentences))
print("First training sentence:", train_sentences[0])
print("Labels:", train_labels[0])

Number of training sentences: 14987
First training sentence: ['-DOCSTART-']
Labels: ['O']


In [12]:
# Convert sentences to text
train_texts = [' '.join(sent) for sent in train_sentences]
valid_texts = [' '.join(sent) for sent in valid_sentences]
test_texts  = [' '.join(sent) for sent in test_sentences]

# Quick check
print("Sample text from train:", train_texts[0])

Sample text from train: -DOCSTART-


In [19]:
model = spacy.load("en_core_web_sm")  # pretrained English NER model

In [20]:
sample_doc = model(train_texts[0])  # first training sentence

print("Named Entities in first sentence:")
for ent in sample_doc.ents:
    print(ent.text, ent.label_)

# Optional visualization in Jupyter/VSCode interactive
displacy.render(sample_doc, style='ent', jupyter=True)

Named Entities in first sentence:


C:\Users\waqas\AppData\Local\Programs\Python\Python312\Lib\site-packages\spacy\displacy\__init__.py:215: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)


In [22]:
all_entities = []

for text in train_texts:
    doc = model(text)
    for ent in doc.ents:
        all_entities.append((ent.text, ent.label_))

import pandas as pd
entities_df = pd.DataFrame(all_entities, columns=['Entity', 'Label'])

# Top 20 frequent entities
print(entities_df['Entity'].value_counts().head(20))

# Count by entity label
print(entities_df['Label'].value_counts())

Entity
1             346
first         327
U.S.          321
6             320
Thursday      281
2             276
two           275
3             259
Wednesday     247
Tuesday       201
three         191
5             189
second        186
7             184
Monday        179
1996-08-28    164
Friday        160
4             157
one           154
Germany       141
Name: count, dtype: int64
Label
CARDINAL       6578
GPE            6158
DATE           5972
PERSON         5949
ORG            4731
NORP           2011
ORDINAL         994
MONEY           634
TIME            424
QUANTITY        424
PERCENT         318
PRODUCT         307
LOC             209
EVENT           165
FAC             160
WORK_OF_ART      87
LAW              77
LANGUAGE         27
Name: count, dtype: int64


TESTING MY OWN SENTENCE

In [23]:
# Example sentence you want to test
test_sentence = "Apple announced the new iPhone in California on Tuesday."

# Process it with spaCy
doc = model(test_sentence)

# Print entities found
print("Named Entities in test sentence:")
for ent in doc.ents:
    print(ent.text, ent.label_)

# Optional visualization
displacy.render(doc, style='ent', jupyter=True)

Named Entities in test sentence:
Apple ORG
California GPE
Tuesday DATE
